### Qwen 2.5 5B Model Architecture

In [ ]:
# Upgrade necessary packages
import importlib.util

if importlib.util.find_spec("google.colab"):
    # install packages into the current Jupyter kernel's Python environment.
    %pip install -q transformers tiktoken transformers_stream_generator bertviz

In [ ]:
# Free GPU Memory
import torch
import gc

# Unload models and clean up gpu memory cache
def free_gpu(model):
  if model:
    # Removes the reference to the model's memory,
    # making it eligible for garbage collection.
    del model

  # Release any cached GPU memory that's no longer needed.
  torch.cuda.empty_cache()

  # Trigger garbage collection to ensure memory is fully released.
  gc.collect()

#### Look at model config, What is model.config? 
It's a configuration object that contains all the hyperparameters and settings of the model It defines the model's architecture, size, and behavior It's essentially a blueprint of how the model is structured

In [ ]:
from transformers import AutoModelForCausalLM
from pprint import pprint
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

model_name = "Qwen/Qwen2.5-0.5B"

# load Qwen 2.5. First time installs to $HF_HOME location.
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    device_map = "auto"
)

# Print all model configuration parameters
config = model.config

print("=== Model Configuration Parameters ===")
# Architecture parameters
print("\nArchitecture Parameters:")
print(f"Hidden size: {config.hidden_size}")  # Size of the hidden layers (dmodel)
print(f"Number of layers: {config.num_hidden_layers}")  # Number of transformer blocks
print(f"Number of attention heads: {config.num_attention_heads}")  # Number of attention heads in each transformer block
print(f"Intermediate size: {config.intermediate_size}")  # Size of the FFN/MLP intermediate layer. FFN (inverted bottleneck) then expands dmodel → config.intermediate_size → dmodel

# Tokenizer parameters
print("\nTokenizer Parameters:")
print(f"Vocabulary size: {config.vocab_size}")  # Size of the vocabulary
print(f"Maximum position embeddings: {config.max_position_embeddings}")  # Maximum sequence length. This is context windows size.

# Print model size
total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel Size:")
print(f"Total parameters: {total_params:,}")

# Model-specific parameters
print("\nModel-specific Parameters:")
for key, value in config.to_dict().items():
    if key not in ['architectures', 'model_type', 'torch_dtype']:
        print(f"{key}: {value}")

# Print memory usage analysis
actual_weight_bytes = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
)

print(f"Actual model weight memory (MB): {actual_weight_bytes / (1024 * 1024):.2f}")

# Free GPU memory
free_gpu(model)


#### Model Architecture
Helps inspect the decoder layer which is crucial when planning model optimization or serving. Attention heavy blocks can benefit from fused attention kernel like optimizations.

In [ ]:

# We are using model object already created before.
print(f"\n=== {model_name} Architecture ===")
print("\nModel Configuration:")
# pprint(model.config.to_dict())

print("\nModel Structure:")
def print_module_structure(module, prefix=''):
    for name, child in module.named_children():
        # Skip certain internal modules for clarity
        if name in ['_orig_mod', 'wrapped_model']:
            continue

        # Print the current module
        print(f"{prefix}{name}: {type(child).__name__}")

        if "Qwen2Attention" in name.lower():
          print(f"\nFound attention module: {name}")
          print(f"Type: {type(module).__name__}")

          # Print attention-specific attributes
          if hasattr(module, 'num_heads'):
              print(f"Number of attention heads: {module.num_heads}")
          if hasattr(module, 'head_dim'):
              print(f"Head dimension: {module.head_dim}")
          if hasattr(module, 'hidden_size'):
              print(f"Hidden size: {module.hidden_size}")
          if hasattr(module, 'rotary_emb'):
              print(f"Has rotary embeddings: {module.rotary_emb is not None}")

        # If it's a container module (has children), recurse
        if list(child.children()):
            print_module_structure(child, prefix + '  ')

print_module_structure(model)


#### Attention Layer
Inspecting the Attention layer in Detail

In [ ]:

# We are using model object already created before.
print(f"\n=== Detailed {model_name} Attention Analysis ===")

# Find all attention layers
attention_layers = []
for name, module in model.named_modules():
    #if "attention" in name.lower():
    if "Qwen2Attention" in type(module).__name__:
        attention_layers.append((name, module))

print(f"\nFound {len(attention_layers)} attention layers")

# Analyze each attention layer
for i, (name, module) in enumerate(attention_layers):
    print(f"\nAttention Layer {i}: {name}")
    print("=" * 50)

    # Basic information
    print(f"Type: {type(module).__name__}")

    # Attention parameters
    if hasattr(module, 'num_heads'):
        print(f"Number of attention heads: {module.num_heads}")
    if hasattr(module, 'head_dim'):
        print(f"Head dimension: {module.head_dim}")
    if hasattr(module, 'hidden_size'):
        print(f"Hidden size: {module.hidden_size}")

    # Rotary embeddings
    if hasattr(module, 'rotary_emb'):
        print(f"Rotary embeddings: {type(module.rotary_emb).__name__ if module.rotary_emb else 'None'}")

    # Attention projections
    print("\nAttention projections:")
    for sub_name, sub_module in module.named_children():
        if hasattr(sub_module, 'weight'):
            shape = sub_module.weight.shape
            print(f"  {sub_name}: {type(sub_module).__name__}, Shape: {shape}")

    # Additional attention-specific attributes
    print("\nAdditional attributes:")
    for attr_name in dir(module):
        if not attr_name.startswith('_') and not callable(getattr(module, attr_name)):
            try:
                value = getattr(module, attr_name)
                if not isinstance(value, (torch.Tensor, torch.nn.Module)):
                    print(f"  {attr_name}: {value}")
            except:
                pass

# Print model's attention-related configuration
print("\nAttention-related configuration:")
config = model.config.to_dict()
attention_config = {k: v for k, v in config.items() if 'attention' in k.lower()}
pprint(attention_config)

#### Attention Head and Tokens
Here we will use the BertViz library's head view to visualize the attention of each layer of Qwen model and how different attention heads in a layer connect different tokens.
Each line represents an attention connection between two tokens - line thickness indicates the strength of the attention weight, while color distinguishes different attention heads. By examining these we can observe which tokens a give token attends to most strongly.

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from pprint import pprint
# prompt: use bertviz library to visualize the attention result of the input prompt "write a short introduction about US capital city"
from transformers import AutoTokenizer
from bertviz import head_view

# Your input text
text = "write a short introduction about the US capital city"
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name, output_attentions=True)
model = AutoModelForCausalLM.from_pretrained(model_name, output_attentions=True).eval().cuda()

# Tokenize input and get token strings
inputs = tokenizer(text, return_tensors="pt").to(model.device)
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

# Generate outputs with attention
with torch.no_grad():
    outputs = model(**inputs, output_attentions=True)

# Get attention weights
attention = outputs.attentions

# Use bertviz to visualize
head_view(attention, tokens)

free_gpu(model)